In [ ]:
from pathlib import Path
import shutil
DATA = Path.cwd() / "data"
DATA.mkdir(exist_ok=True)
expected = 'house_price_prediction question two.xlsx'
if not (DATA / expected).exists():
    from google.colab import files
    print("Upload:", expected)
    uploaded = files.upload()
    if expected not in uploaded:
        raise ValueError("Please upload the file with its original name: " + expected)
    (DATA / expected).write_bytes(uploaded[expected])
print("Dataset ready:", expected)


## Execute the solution
The comments identify the operations and question parts.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path.cwd()
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)
# Row 3 contains headers; use raw inputs only, not Excel predictions.
df = pd.read_excel(BASE / "data/house_price_prediction question two.xlsx",
                   sheet_name="House Prices", header=2,
                   nrows=15, usecols="A:C")
X = df[["Size (m²)"]]
y = df["Price (UGX millions)"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
results = pd.DataFrame({
    "House": df.loc[X_test.index, "House"],
    "Size_m2": X_test["Size (m²)"],
    "Actual_UGX_millions": y_test,
    "Predicted_UGX_millions": predictions,
    "Residual_actual_minus_predicted": y_test - predictions
}).sort_values("House")
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
new_size = pd.DataFrame({"Size (m²)": [130]})
new_prediction = model.predict(new_size)[0]
print("Training records:", len(X_train), "Test records:", len(X_test))
print(f"Price = {model.intercept_:.6f} + {model.coef_[0]:.6f} * size")
print(results.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"Test MAE: {mae:.3f} UGX millions")
print(f"Test RMSE: {rmse:.3f} UGX millions")
print(f"Test R-squared: {r2:.6f}")
print(f"Prediction for 130 m²: {new_prediction:.3f} UGX millions")
results.to_csv(OUT / "q2_test_predictions.csv", index=False)
metrics = dict(mae=float(mae), rmse=float(rmse), r2=float(r2),
               slope=float(model.coef_[0]), intercept=float(model.intercept_),
               prediction_130=float(new_prediction))
(OUT / "q2_metrics.json").write_text(json.dumps(metrics, indent=2))


Training records: 12 Test records: 3
Price = 30.763663 + 2.036189 * size
 House  Size_m2  Actual_UGX_millions  Predicted_UGX_millions  Residual_actual_minus_predicted
     1       50                  128                 132.573                           -4.573
    10      140                  318                 315.830                            2.170
    12      160                  362                 356.554                            5.446
Test MAE: 4.063 UGX millions
Test RMSE: 4.293 UGX millions
Test R-squared: 0.998213
Prediction for 130 m²: 295.468 UGX millions


## Interpretation
The held-out MAE is approximately 4.063 UGX million and RMSE is 4.293 UGX million. Test R² is about 0.998213. These strong results concern only three test records from constructed teaching data; they do not establish accuracy for the Ugandan housing market.

## Download generated files
Run this cell in Colab to download the results ZIP. Save your notebook too using **File → Download → Download .ipynb**.

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path
archive = Path("question_2_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in Path("outputs").glob("*"):
        if path.is_file() and (path.name.startswith("q2_") or (2 == 4 and path.name == "student_records.json")):
            z.write(path)
    if 2 == 3:
        z.write("animals.py")
        z.write("q3_main.py")
        z.writestr("q3_output.txt", result.stdout)
files.download(str(archive))
